# 1st LAB: Word Embeddings

In [1]:
import re
import string

import nltk
import numpy as np
import pandas as pd

# Loading the data

In [7]:
# US presidential inaugural speeches, from the NLTK corpora
nltk.download('inaugural') ###
from nltk.corpus import inaugural

[nltk_data] Downloading package inaugural to /root/nltk_data...
[nltk_data]   Package inaugural is already up-to-date!


In [16]:
# Extract the year, president's name and speech text from each file
data = pd.DataFrame({
    # Extract the year from the first 4 characters of each filename
    'year': [int(f[:4]) for f in inaugural.fileids()],
    # Extract the president's name from 6th char to last before ".txt"
    'president': [f[5:-4] for f in inaugural.fileids()],
    # Retrieve the full text of each inaugural speech
    'text': [inaugural.raw(f) for f in inaugural.fileids()],
}).sort_values('year', ignore_index=True)  # Sort speeches by year

data.head()

,year,president,text
0,1789,Washington,Fellow-Citizens of the Senate and of the House...
1,1793,Washington,"Fellow citizens, I am again called upon by the..."
2,1797,Adams,"When it was first perceived, in early times, t..."
3,1801,Jefferson,Friends and Fellow Citizens:\n\nCalled upon to...
4,1805,Jefferson,"Proceeding, fellow citizens, to that qualifica..."


In [17]:
# Define functions to remove punctuation and numbers from text
def remove_punctuation(text):
    return re.sub(f"[{re.escape(string.punctuation)}]", " ", text)

def remove_numbers(text):
    return re.sub(r"\d+", "", text)

In [21]:
# Convert text to lowercase
data["preprocessed"] = data["text"].str.lower()

# Remove numbers and punctuation
data["preprocessed"] = data["preprocessed"].apply(remove_numbers)
data["preprocessed"] = data["preprocessed"].apply(remove_punctuation)

# Split text into individual words (tokens)
data["tokens"] = data["preprocessed"].str.split()

data.head()

,year,president,text,preprocessed,tokens
0,1789,Washington,Fellow-Citizens of the Senate and of the House...,fellow citizens of the senate and of the house...,"[fellow, citizens, of, the, senate, and, of, t..."
1,1793,Washington,"Fellow citizens, I am again called upon by the...",fellow citizens i am again called upon by the...,"[fellow, citizens, i, am, again, called, upon,..."
2,1797,Adams,"When it was first perceived, in early times, t...",when it was first perceived in early times t...,"[when, it, was, first, perceived, in, early, t..."
3,1801,Jefferson,Friends and Fellow Citizens:\n\nCalled upon to...,friends and fellow citizens \n\ncalled upon to...,"[friends, and, fellow, citizens, called, upon,..."
4,1805,Jefferson,"Proceeding, fellow citizens, to that qualifica...",proceeding fellow citizens to that qualifica...,"[proceeding, fellow, citizens, to, that, quali..."


In [23]:
print(len(data))

60


# Exercise 3.1: Static Word Embeddings

---




Steps:

Part 1 **Training Word2vec on a dataset**:
- Train a Word2Vec model on the presidents' speeches. You must pass to Word2Vec a list of lists of tokens (a list of tokenized sentences). Use CBOW and a window of 5. Documentation: https://radimrehurek.com/gensim/models/word2vec.html#gensim.models.word2vec.Word2Vec
- Find the words most similar to, e.g. "state", "states", "president", or others.
- Compare the effectiveness of CBOW and Skipgram. Which one works better?


Part 2 **Using pre-trained static models**:
- Load a pre-trained gensim model (available models: https://github.com/RaRe-Technologies/gensim-data#Models) (Choose a small one to avoid waiting for the download, e.g. `glove-wiki-gigaword-100`).
- Evaluate the following relations:
  - king + woman - man = ?
  - France - Paris + London = ?
  - Find out some for yourselves

**Question: Find the words most similar to "italy" using the 2 models (trained and pretrained). What do you observe? Why? Any solution?**



In [ ]:
# Your code here

from gensim import downloader
from gensim.models import Word2Vec

In [ ]:
nltk.download('punkt') ###< run once
nltk.download('punkt_tab') ###< run once, needed by recent NLTK versions

In [ ]:
data['sent_tokens'] = data.text.str.lower()

data.sent_tokens = data.sent_tokens.str.replace(r'\d+', ' ', regex=True)
data.sent_tokens = data.sent_tokens.apply(lambda input_text : [remove_punctuation(t).split() for t in nltk.sent_tokenize(input_text)])

In [ ]:
print('Number of sentences:', len(data.sent_tokens.sum()))

In [ ]:
print('Number of speeches: ', len(data.sent_tokens))

In [ ]:
corpus = data.sent_tokens.sum() # set a list of sentences rather than list of list of words
# now use the slides code for training!

# sg=0 for CBOW, sg=1 for skip-gram
cbow_model = Word2Vec(sentences=corpus, vector_size=100, window=5, sg=0, seed=42)
model = Word2Vec(sentences=corpus, vector_size=100, window=5, sg=1, seed=42)

# set similar words to:
# states, war, state
print('CBOW:     ', [word for word, score in cbow_model.wv.most_similar('states', topn=8)])
print('Skip-gram:', [word for word, score in model.wv.most_similar('states', topn=8)])

# try also italy and see what happens!

In [ ]:
# 'italy' never appears in the speeches, so it has no vector
print("Is 'italy' in the vocabulary?", 'italy' in model.wv)

In [ ]:
downloader.info()['models'].keys()

dict_keys(['fasttext-wiki-news-subwords-300', 'conceptnet-numberbatch-17-06-300', 'word2vec-ruscorpora-300', 'word2vec-google-news-300', 'glove-wiki-gigaword-50', 'glove-wiki-gigaword-100', 'glove-wiki-gigaword-200', 'glove-wiki-gigaword-300', 'glove-twitter-25', 'glove-twitter-50', 'glove-twitter-100', 'glove-twitter-200', '__testing_word2vec-matrix-synopsis'])

In [ ]:
wiki_model = downloader.load('glove-wiki-gigaword-100')

[==================================================] 100.0% 128.1/128.1MB downloaded


In [ ]:
president_vector = wiki_model['president']
print('Vector size:', president_vector.shape)
print('First 5 values:', president_vector[:5])

In [ ]:
wiki_model.most_similar(positive=['king', 'woman'], negative=['man'], topn=5)

[('queen', 0.7698541283607483),
 ('monarch', 0.6843380928039551),
 ('throne', 0.6755735874176025),
 ('daughter', 0.6594556570053101),
 ('princess', 0.6520534753799438)]

In [ ]:
wiki_model.most_similar(positive=['france', 'london'], negative=['paris'], topn=5)

[('britain', 0.8921419978141785),
 ('england', 0.8002905249595642),
 ('australia', 0.7940487861633301),
 ('scotland', 0.7838967442512512),
 ('british', 0.7608305215835571)]

In [ ]:
# 'italy' is not in our speeches, but it is in the pre-trained model (trained on Wikipedia)
wiki_model.most_similar('italy', topn=5)

# Exercise 3.2: Embeddings with multiword ngrams

---



**Training Word2vec on a dataset**:
- Train a bigram detector using gensim.
- Train a Word2Vec model on the presidents' speeches. You must pass to Word2Vec a list of lists of bigram tokens (a list of tokenized sentences).
- Find the words most similar to, e.g. "united_states", "american_people" or others.
- Try both word2vec architectures.

**Have you seen the sparsity problem?**

In [ ]:
from gensim.models import Phrases

# train the bigram detector on the sentences (the same corpus as in Exercise 3.1)
bigram_generator = Phrases(corpus)

In [ ]:
bigram_tokens = list(bigram_generator[corpus])

# how many times does each bigram appear? (fewer occurrences = less to learn from)
bigram_counts = pd.Series([w for sentence in bigram_tokens for w in sentence]).value_counts()
print(bigram_counts[['united_states', 'american_people']])

In [ ]:
# use bigram_tokens as your new corpus (same code as before!)
model = Word2Vec(sentences=bigram_tokens, vector_size=100, window=5, sg=1, seed=42)

In [ ]:
print(model.wv.most_similar('united_states', topn=10))

[('constitution', 0.9667254090309143), ('states', 0.9621188044548035), ('whole', 0.9528996348381042), ('rights', 0.9472708106040955), ('powers', 0.9415111541748047), ('executive', 0.9414318799972534), ('laws', 0.9409316778182983), ('general_government', 0.9398400187492371), ('interests', 0.9381714463233948), ('union', 0.9362156391143799)]


In [ ]:
print(model.wv.most_similar('american_people', topn=10))

[('second', 0.9975712299346924), ('destinies', 0.9974365234375), ('encouragement', 0.9973673224449158), ('officer', 0.9972285032272339), ('assume', 0.9971164464950562), ('expressed', 0.9969063401222229), ('same', 0.9968450665473938), ('source', 0.9966413974761963), ('direction', 0.9964177012443542), ('lot', 0.996334433555603)]


# Exercise 3.3

You can represent a document as the weighted sum of the word embeddings of the terms it contains, where the term weights are given by, e.g. Tf-Idf.

- Compute the document embeddings for the US presidential speeches corpus.
- Write a function that, given a document (speech) id, finds the top n most similar documents (adapt the previous one for most similar terms).


Hints (but there may be other / better solutions):

- To obtain the TfIdf values from the 'tfidf' matrix returned by fit_transform, use 'tfidf[doc_id, term_index]'. The list of terms (features) is returned by 'vectorizer.get_feature_names_out()'
- To obtain the Word2Vec embedding for a word, use 'model.wv[word]', where 'model = Word2Vec(...)'
- To obtain the document embedding, iterate over all of its words (that are present both as features for the TfIdf vectorizer and in the Word2Vec model vocabulary), and sum over all the products: (model.wv[word] * tfidf[doc_id, word_index])

In [ ]:
# Your code here

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

vectorizer = TfidfVectorizer()
docs_tfidf = vectorizer.fit_transform(data.preprocessed)

model = Word2Vec(sentences=corpus, window=5, sg=1, seed=42) # we use skipgram

In [ ]:
tf_idf_names = vectorizer.get_feature_names_out()

In [ ]:
skip_gram_words = model.wv.key_to_index

In [ ]:
tfidf_feature_to_index = {word: i for i, word in enumerate(tf_idf_names)}

def tfidf_weight(doc_id, term):
    return docs_tfidf[doc_id, tfidf_feature_to_index[term]]

In [ ]:
skip_gram_set = set(skip_gram_words)
tfidf_set = set(tf_idf_names)

common_terms = skip_gram_set.intersection(tfidf_set)

In [ ]:
def get_document_embedding(doc_id, common_terms):
    document = data.tokens[doc_id]
    embedding = np.zeros(model.vector_size)
    for word in set(document):   # each word once: the TF-IDF weight already counts its repetitions
        if word in common_terms:
            embedding = embedding + model.wv[word] * tfidf_weight(doc_id, word)

    return embedding

In [ ]:
doc_embeddings = np.zeros((data.shape[0], model.vector_size))

for i in range(data.shape[0]):
    doc_embeddings[i] = get_document_embedding(i, common_terms)

In [ ]:
doc_embeddings.shape

(56, 100)

In [ ]:
doc_cos = cosine_similarity(doc_embeddings, doc_embeddings)

def top_n_most_similar_document(doc_id, doc_sim, top_n=5):
    current_doc_sim = doc_sim[doc_id]
    best = (-current_doc_sim).argsort()[1:(top_n+1)]   # position 0 is the document itself
    return data[['year', 'president']].loc[best].assign(similarity=current_doc_sim[best].round(3))

In [ ]:
doc_cos.shape

(56, 56)

In [ ]:
data.loc[2, ['year', 'president']]

In [ ]:
top_n_most_similar_document(2, doc_cos, 5)

president  year  \
5   Jackson   1833   
1   Harrison  1889   
47  Taft      1909   
49  McKinley  1897   
41  Hayes     1877   

                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                              

In [ ]:
# Are the speeches really different? Look at the similarities between different speeches
others = doc_cos[~np.eye(len(doc_cos), dtype=bool)]   # all the values except the diagonal
print('Min similarity: ', others.min().round(3))
print('Mean similarity:', others.mean().round(3))

All the speeches look almost identical to each other: they all use the same frequent words, so the sums of their vectors point in almost the same direction.

A simple fix: subtract the **mean vector** of all the speeches. What is left is what makes each speech different from the others.

In [ ]:
doc_embeddings_centered = doc_embeddings - doc_embeddings.mean(axis=0)
doc_cos_centered = cosine_similarity(doc_embeddings_centered, doc_embeddings_centered)

others = doc_cos_centered[~np.eye(len(doc_cos_centered), dtype=bool)]
print('Min similarity: ', others.min().round(3))
print('Mean similarity:', others.mean().round(3))

# Compare the most similar speeches to Biden 2021, before and after centering
biden_id = data.index[data.year == 2021][0]
display(top_n_most_similar_document(biden_id, doc_cos))
display(top_n_most_similar_document(biden_id, doc_cos_centered))

# Exercise 3.4

Use Doc2Vec (https://radimrehurek.com/gensim/models/doc2vec.html) to compute the embeddings for the documents in the presidential speeches dataset. Write a function that finds the most similar documents for a given document(adapt the previous function).

In [ ]:
# Your code here

from gensim.models.doc2vec import Doc2Vec, TaggedDocument

In [ ]:
documents = [TaggedDocument(doc, [i]) for i, doc in enumerate(data.tokens)]

In [ ]:
doc_model = Doc2Vec(documents, window=5, seed=42)

In [ ]:
# generic method to get an emebedding not based only on the document id
# d2v_doc_embeddings = np.array([doc_model.infer_vector(data.tokens[i]) for i in range(data.shape[0])])


# use the static embedding generated by doc2vec model
d2v_doc_embeddings = np.array([doc_model.dv[i] for i in range(data.shape[0])])

In [ ]:
d2v_doc_embeddings.shape

(56, 100)

In [ ]:
d2v_doc_cos = cosine_similarity(d2v_doc_embeddings, d2v_doc_embeddings)

In [ ]:
top_n_most_similar_document(2, d2v_doc_cos)

president  year  \
43  Madison    1809   
31  Hayes      1877   
22  Adams      1797   
54  Monroe     1817   
46  Cleveland  1885   

                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                         

In [ ]:
ex_query = 'fellow citizens of the united states of america'
# infer the query (or a doc) vector using only the words (so good for new documents or queries)
# infer_vector wants a list of words, not a string
q = doc_model.infer_vector(ex_query.split())

# print(cosine_similarity(d2v_doc_embeddings, q.reshape(1, -1)))
cosine_similarity(d2v_doc_embeddings, q.reshape(1, -1)).shape

In [ ]:
def get_top_n_from_query(query, doc_embeddings, top_n=5):
    q = doc_model.infer_vector(query.split())
    similarities = cosine_similarity(doc_embeddings, q.reshape(1, -1)).flatten()
    # one row per speech, sorted from the most similar
    ranking = data[['year', 'president']].assign(similarity=similarities.round(3))
    return ranking.sort_values('similarity', ascending=False).head(top_n)

# infer_vector is random: we save the result to use it again below
top_docs = get_top_n_from_query(ex_query, d2v_doc_embeddings)
top_docs

In [ ]:
ex_query

'fellow citizien of the united states of america'

In [ ]:
# beginning of the most similar speech
top_id = top_docs.index[0]
print(data.loc[top_id, 'year'], data.loc[top_id, 'president'])
print(data.loc[top_id, 'text'][:200], '...')